# Import package

In [5]:
!pip install tenacity
!pip install pdfminer

  Created wheel for pdfminer: filename=pdfminer-20191125-py3-none-any.whl size=6140114 sha256=60da730c0cc74ea204f14898a339c602e0d4a6b94c7d09ef15f231649b4abcfe
  Stored in directory: c:\users\memphis meng\appdata\local\pip\cache\wheels\e3\5e\f4\d210b46e9e4a28229ea070ed5b3efa92c3c29d1a7918dd4b97
Successfully built pdfminer


In [10]:
from bs4 import BeautifulSoup as bs     # parsing
from urllib.request import urlopen      # scrapping
import pandas as pd                     # exporting
from tenacity import retry, stop_after_attempt
import http.client

In [11]:
from io import StringIO
from pdfminer.converter import TextConverter
from pdfminer.layout import LAParams
from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
from pdfminer.pdfpage import PDFPage
import os
import logging
from tqdm import tqdm
import csv
import re

In [12]:
address_1 = re.compile('(?:ONE|\d{1,4})?[\w\s\W]{1,20}(?:STREET|ST|SRTEET|STR|AVENUE|AVE|ROAD|RD|HIGHWAY|HWY|SQUARE|SQ|TRAIL|TRL|DRIVE|DR|COURT|CT|PARK|PARKWAY|PKWY|CIRCLE|CIR|BOULEVARD|BLVD|TERRACE|TERR|TER|LANE|LN|HL|PATH|WAY|PLACE|BEACH|ROUTE)?\.?\W?(?=\s|$)', re.IGNORECASE)
address_2 = re.compile('(?:ONE|\d{1,4})[\w\s\W]{1,20}(?:STREET|ST|SRTEET|STR|AVENUE|AVE|ROAD|RD|HIGHWAY|HWY|SQUARE|SQ|TRAIL|TRL|DRIVE|DR|COURT|CT|PARK|PARKWAY|PKWY|CIRCLE|CIR|BOULEVARD|BLVD|TERRACE|TERR|TER|LANE|LN|HL|PATH|WAY|PLACE|BEACH|ROUTE)?\.?\W?(?=\s|$)', re.IGNORECASE)

city = re.compile('[\w\s]{1,20}[,] \w{1,}?[\d\W]{1,11} [\w\s]{1,3}')
date = re.compile('(?:(?<!\:)(?<!\:\d)[0-3]?\d(?:st|nd|rd|th)?\s+(?:of\s+)?(?:jan\.?|january|feb\.?|february|mar\.?|march|apr\.?|april|may|jun\.?|june|jul\.?|july|aug\.?|august|sep\.?|september|oct\.?|october|nov\.?|november|dec\.?|december)|(?:jan\.?|january|feb\.?|february|mar\.?|march|apr\.?|april|may|jun\.?|june|jul\.?|july|aug\.?|august|sep\.?|september|oct\.?|october|nov\.?|november|dec\.?|december)\s+(?<!\:)(?<!\:\d)[0-3]?\d(?:st|nd|rd|th)?)(?:\,)?\s*(?:\d{4})?|[0-3]?\d[-\./][0-3]?\d[-\./]\d{2,4}', re.IGNORECASE)
number = re.compile('\d{1,20}')
title = ['PRESIDENT', 'VICE PRESIDENT', 'VICE PRESIDENT, TAX', 'TREASURER', 'ASSISTANT TREASURER', 'ASST. TREASURER', 'ASSISTANT CLERK', 'CLERK',
         'SECRETARY', 'ASSISTANT SECRETARY', 'CORPORATE SECRETARY', 'GENERAL SECRETARY', 'DIRECTOR', 'SECRETARY & DIRECTOR', 'MEDICAL DIRECTOR',
         'ASST TREAS/SECY', 'EDUCATION DIRECTOR', 'CO-WARDEN', 'VICE PRESIDENT FOOTBALL', 'VICE PRESIDENT CHEER', 'REGISTRAR', 'PUBLIC RELATIONS',
         'SENIOR PASTOR', 'ASSISTANT PASTOR', 'CHURCH LEADER', 'COLLECTOR', 'MODERATOR', 'CHAIR', 'GENERAL MEMBER', 'CEO', 'CFO',
         'GENERAL EVANGELIST', 'EXECUTIVE MINISTER', 'CORP CONTROLLER', 'CORPORATE CONTROLLER', 'OFFICER', 'CHAIRMAN & CEO', 'MANAGER',
         'PRESIDENT & CHIEF OPERATING', 'OPERATIONS', 'EDUCATION', 'CHAIRMAN', 'ICE DIRECTOR', 'CHIEF FINANCIAL OFFICER', 'DIRECTOR OF FINANCE',
         'CHIEF EXECUTIVE OFFICER', 'CHIEF OPERATING OFFICER', 'CHIEF COMPLIANCE OFFICER', 'IMMEDIATE PAST CHAIR', 'CHAIR-ELECT',
         'DIRECTOR-CHAIRMAN', 'CFO-TREASURER', 'EVP-COO', 'SENIOR MINISTER', 'MISSIONARY', 'ARCHIVIST']

# Util functions

In [13]:
def process_pdf(rsrcmgr, device, fp, pagenos=None, maxpages=0, password='', caching=True, check_extractable=True):
    interpreter = PDFPageInterpreter(rsrcmgr, device)
    for page in PDFPage.get_pages(fp, pagenos, maxpages=maxpages, password=password,
                                  caching=caching, check_extractable=check_extractable):
        interpreter.process_page(page)
    return

In [14]:
def read_pdf(pdf):
    rsrcmgr = PDFResourceManager()
    retstr = StringIO()
    laparams = LAParams()
    device = TextConverter(rsrcmgr, retstr, laparams=laparams)
    process_pdf(rsrcmgr, device, pdf)
    device.close()
    content = retstr.getvalue()
    retstr.close()
    lines = str(content).split("\n")
    return lines

In [15]:
def write_csv(content, path):
    with open(path, 'w') as f:
        w = csv.writer(f)
        w.writerows(content.items())

In [16]:
def handle(rst, phase):
    '''
    param rst: str, the object representing the info extracted from the pdf, usually the result from function read_pdf
    param phase: int, the phase protocol, to signal which regex compiler should be in use
    '''
#     print(rst)
    idnum = ''
    exactName = ''
    jurisdiction = ''
    registeredName = ''
    registeredAddress = ''
    principalAddress = ''
    directors_info = []
    business = ''
    address = None
    
    if phase == 1:
        address = address_1
    elif phase == 2:
        address = address_2
    
    for i in range(len(rst)):
        if 'Identification Number' in rst[i]:
            idnum = rst[i].split(': ')[1].strip()
        
        if 'Exact name' in rst[i] or 'exact name' in rst[i]:
            exactName = rst[i].split(': ')[1].lstrip()
                
            count = i + 1
            tmp = rst[count]
            while 'Jurisdiction of Incorporation' not in tmp and 'registered office' not in tmp and \
                  'principal office' not in tmp:
                exactName += tmp.strip()
                count += 1
                tmp = rst[count]
            
            exactName = exactName.split('2.')[0].strip()
            
        if 'Jurisdiction of Incorporation' in rst[i]:
            state = ''
            country = ''
            count = i
            tmp = rst[count]
            while 'registered office' not in tmp and 'principal office' not in tmp:
                if 'State' in tmp:
                    state = tmp.split('State: ')[1][0:2]
                if 'Country' in tmp:
                    country = tmp.split('Country: ')[1].strip()
                count += 1
                tmp = rst[count]

            if len(country) != 0:
                jurisdiction = state + ', ' + country
            else:
                jurisdiction = state
            
        if 'corporation registered office' in rst[i]:
            registeredStreetList = []
            registeredStreet = ''
            registeredCity = ''
            registeredState = ''
            registeredZip = ''
            registeredCountry = ''
            count = i + 1
            tmp = rst[count]
            while 'principal office' not in tmp and 'directors' not in tmp:
                if (address.search(tmp) != None) and (tmp.isupper()):
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        registeredStreetList.append(tmp.strip())
                    else:
                        registeredStreetList.append(tmp.split(',')[0].strip())
                elif registeredName == '':
                    if tmp.isupper():
                        registeredName = tmp.strip()
                elif tmp.isupper() and tmp.replace(' ', '').replace('.', '').isalpha():
                    registeredCity = tmp.strip()
                    
                if 'State' in tmp:
                    try:
                        registeredState = tmp.split('State: ')[1].strip()
                    except:
                        registeredState = ''
                if 'Zip' in tmp:
                    try:
                        registeredZip = re.findall(r'\b\d{5}(?:[-\s]\d{4})?\b', tmp)[0]
                    except:
                        registeredZip = ''
                if 'Country' in tmp:
                    registeredCountry = tmp.split('Country: ')[1].strip()
                    
                count += 1
                tmp = rst[count]
            
            for i in range(len(registeredStreetList)):
                registeredStreet += registeredStreetList[i]
                registeredStreet += ', '
            
            registeredAddress = registeredStreet + registeredCity + ', ' + registeredState + \
                                ' ' + registeredZip + ' ' + registeredCountry
            
        if 'principal office' in rst[i]:
            principalStreetList = []
            principalStreet = ''
            principalCity = ''
            principalState = ''
            principalZip = ''
            principalCountry = ''
            count = i + 1
            tmp = rst[count]
            while 'directors' not in tmp:
                if tmp.isupper() and '3.' not in tmp:
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        principalStreetList.append(tmp.strip())
                    else:
                        principalStreetList.append(tmp.split(',')[0].strip())
                
                if 'State' in tmp:
                    try:
                        principalState = tmp.split(': ')[1].strip()
                    except:
                        principalState = ''
                if 'Zip' in tmp:
                    try:
                        principalZip = re.findall(r'\b\d{5}(?:[-\s]\d{4})?\b', tmp)[0]
                    except:
                        principalZip = ''
                if 'Country' in tmp:
                    principalCountry = tmp.split(': ')[1].strip()
                
                count += 1
                tmp = rst[count]
            
            for i in range(len(principalStreetList)):
                principalStreet += principalStreetList[i]
                principalStreet += ', '
            
            principalAddress = principalStreet + principalState + \
                               ' ' + principalZip + ' ' + principalCountry
            
        if 'Title' in rst[i]:
            directorsTitle = []
            directorsName = []
            directorsStreet = []
            directorsCity = []
            count = i + 1
            tmp = rst[count]
            while 'Briefly describe' not in tmp and 'cemetery corporation' not in tmp and 'undersigned' not in tmp:
                # sometimes the title name varies, we can just check if it contains any one in the title list
                if tmp.replace('\xa0', '').strip() in title or \
                bool([t for t in title if (t in tmp.replace('\xa0', '').strip())]): 
                    directorsTitle.append(tmp.strip())
                elif ((city.search(tmp) != None) and (tmp.isupper())) or ('CITY' in tmp):
                    directorsCity.append(tmp.strip())
                
                # usually the column of name is on the left of the address, so this one step should go first
                elif (tmp.isupper()) and (len(tmp.split()) > 1) and \
                     (re.sub(r'[^\w\s]','',tmp).replace('\xa0', '').replace(' ', '').isalpha()):
                    if address.search(tmp) == None:
                        directorsName.append(tmp.strip())
                    elif ',' in tmp and tmp.strip().split(',')[1] != '':
                        directorsStreet.append(tmp.strip())
                    else:
                        directorsStreet.append(tmp.split(',')[0].strip())
                        
                elif ((address.search(tmp) != None) and (tmp.isupper())) or ('PO BOX' in tmp) or (tmp.strip() == 'SAME'):
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        directorsStreet.append(tmp.strip())
                    else:
                        directorsStreet.append(tmp.split(',')[0].strip())
                    
                count += 1
                tmp = rst[count]
            
            if len(directorsCity) == 0:
                while 'Check' not in tmp:
                    if ((city.search(tmp) != None) and (tmp.isupper())) or ('CITY' in tmp):
                        directorsCity.append(tmp.strip())
                    elif ((address.search(tmp) != None) and (tmp.isupper())) or ('PO BOX' in tmp):
                        directorsStreet.append(tmp.split(',')[0].strip())
                        
                    count += 1
                    tmp = rst[count]
                
            
            directors_info = [[] for _ in range(len(directorsTitle))]

            if len(directorsTitle) == len(directorsName) and len(directorsName) == len(directorsStreet) and \
                len(directorsStreet) == len(directorsCity):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + \
                                             directorsStreet[i] + ', ' + directorsCity[i])
            elif len(directorsTitle) == len(directorsName) and len(directorsName) == len(directorsStreet):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + \
                                             directorsStreet[i])
            elif len(directorsTitle) == len(directorsName) and len(directorsName) == len(directorsCity):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + directorsCity[i])
            elif len(directorsStreet) == len(directorsName) and len(directorsName) == len(directorsCity):
                for i in range(len(directorsStreet)):
                    directors_info[i].append(directorsName[i] + ', ' + \
                                             directorsStreet[i] + ', ' + directorsCity[i])
            elif len(directorsTitle) == len(directorsName):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i])
            elif len(directorsTitle) == len(directorsCity):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsCity[i])
            elif len(directorsTitle) == len(directorsStreet):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsStreet[i])
            elif len(directorsCity) == len(directorsName):
                for i in range(len(directorsName)):
                    directors_info[i].append(directorsName[i] + ', ' + directorsCity[i])
            elif len(directorsCity) == len(directorsStreet):
                for i in range(len(directorsCity)):
                    directors_info[i].append(directorsStreet[i] + ', ' + directorsCity[i])
            elif len(directorsName) == len(directorsStreet):
                for i in range(len(directorsName)):
                    directors_info[i].append(directorsName[i] + ', ' + directorsStreet[i])
                
        if 'The general character' in rst[i] or 'Briefly describe' in rst[i]:
            count = i + 1
#             tmp = rst[count]
            while 'cid' in tmp or tmp == ' ' or tmp == '' or 'service, the service to be rendered:' in tmp or \
                address.search(tmp) != None or city.search(tmp) != None or tmp.isupper() == False:
                count += 1
                tmp = rst[count]
                break
            business = tmp.strip()
            
    try:
        id = format(int(idnum), '09d') if len(idnum) <= 9  and len(idnum) else None
    except:
        id = idnum
                    
    rec = {

        'ID Number': id, # fixed length = 9

        'Business Name': exactName,

        'Jurisdiction': jurisdiction,
        
        'Registered Office Name': registeredName,
        
        'Registered Office Address': registeredAddress,
        
        'Principal Office Address': principalAddress,
        
        'Directors': directors_info,
        
        'Business': business
        
        }
    
    return rec

In [8]:
path = '.\PDF_data\PDF'
dirs = os.listdir(path)
# business = {}
results = [] 
for file in tqdm(dirs):
    pdf = path + '/' + file
    with open(pdf, 'rb') as f:
        try:
            rst = read_pdf(f)
        except:
            continue
        try:
            info = handle(rst, 1)
            # use the file name as the ID number
            if info['ID Number'] is None:
                info['ID Number'] = format(int(file[:-4]), '09d')
            results.append(info)
        except IndexError:
            # print('list index out of range')
            continue
        except ValueError:
            info = handle(rst, 1)
            # use the file name as the ID number
            if info['ID Number'] is None:
                info['ID Number'] = file[:-4]
            results.append(info)
            continue

100%|██████████| 13152/13152 [50:33<00:00,  4.34it/s] 


In [9]:
resultsDF = pd.DataFrame(results)
resultsDF.to_csv('~/Desktop/sec_results.csv')

# Polishment

In [18]:
result = pd.read_csv('~/Desktop//sec_results.csv', index_col=0)

scanned = result[(result['Business Name'].isna()) & (result['Jurisdiction'].isna()) & (result['Registered Office Name'].isna())
      & (result['Principal Office Address'].isna()) & (result['Business'].isna())]['ID Number']

import os
import shutil


for file in tqdm(scanned):
    pdf = path + '/' + file + '.pdf'
    if os.path.exists(pdf):
        shutil.move(pdf, './../Desktop/pdf1-unextractable' + '/' + file + '.pdf')
        
result.dropna(subset=['Business Name', 'Jurisdiction', 'Registered Office Name', 'Principal Office Address', 'Business'], how='all').\
reset_index(drop=True).to_csv('~/Desktop/sec_results1.csv')

result = pd.read_csv('~/Desktop//sec_results1.csv', index_col=0)

for file in tqdm(result['ID Number']):
    pdf = path + '/' + format(file, '09d') + '.pdf'
    if os.path.exists(pdf):
        shutil.move(pdf, './../Desktop/pdf1-extracted' + '/' + format(file, '09d') + '.pdf')

0it [00:00, ?it/s]
100%|██████████| 412/412 [00:00<00:00, 896.35it/s]


In [64]:
result[result["Directors"].str.contains("\[\]")]

,ID Number,Business Name,Jurisdiction,Registered Office Name,Registered Office Address,Principal Office Address,Directors,Business
16877,000111255,ALBINA TAVARES ASSOCIATION INC.,NaN,NaN,NaN,"96 MAXFIELD ST, NEW BEDFORD, MA 02740 USA","[[], [], [], [], [], [], [], []]",NaN
16892,000000053,"INTERNUTRIA, INC.","DE, USA","PRENTICE-HALL CORPORATION SYSTEM, INC., THE","84 STATE ST., BOSTON, MA 02109 USA","33 HAYDEN AVE, LEXINGTON, MA 02421 USA","[['GLENN L. COOPER M.D., LEXINGTON, MA 02421 U...",INACTIVE
16909,000627115,"APPLUS TECHNOLOGIES, INC.","DE, USA",COGENCY GLOBAL INC.,"45 SCHOOL STREET, STE 202, BOSTON, MA","120 S. LASALLE ST., SUITE 1450, CHICAGO, IL 60...","[[], [], [], [], [], [], [], [], []]",VEHICLE EMISS PROGRAM AND MGMT RELATED PRODUCT...
16925,000000069,A-CS TRANSPORATION INC.,MA,MATTAPAN,"TYRONE THORNTON, 31 HALBORN ST, , MA 02126 USA","31 HALBORN ST., BOSTON, MA 02126 USA","[[], [], [], []]",TRANSPORTATION
16936,000747749,"AIM SYSTEMS, INC.","GA, USA",C T CORPORATION SYSTEM,"155 FEDERAL ST., BOSTON, MA 02110 USA","11675 RAINWATER DR., SUITE 200, ALPHARETTA, GA...","[[], [], [], [], []]",SALE AND PROCESSING OF LIFE INSURANCE
...,...,...,...,...,...,...,...,...
225,586237000,"ALLIANCE COMMERCIAL PROPERTIES, LTD.","DE, USA","UNITED CORPORATE SERVICE,INC","9 CRESTWAY RD.,EAST, BOSTON, MA 02128 USA","165 SO. UNION BLVD.,STE. #510, LAKEWOOD, CO 80...","[[], []]",REAL ESTATE INVESTMENT SERVICES
233,760487432,"ADMINISTAFF COMPANIES, INC.","DE, USA",CORPORATION SERVICE COMPANY,"84 STATE STREET, BOSTON, MA 02109 USA","19001 CRESCENT SPRINGS DR., KINGWOOD, TX 77339...","[[], [], [], [], [], []]",PROFESSIONAL EMPLOYER ORGANIZATION
235,841678925,"ACULAB, INC.","DE, USA",CT CORPORATION SYSTEM,"155 FEDERAL STREET, SUITE 700, BOSTON, MA","100 RIVER RIDGE DRIVE, SUITE 101, NORWOOD, MA ...","[[], [], [], []]",ACULAB PLC
241,942964150,"ABM SECURITY SERVICES, INC.","CA, USA",C T CORPORATION SYSTEM,"155 FEDERAL STREET, STE 700, BOSTON, MA","3800 BUFFALO SPEEDWAY, SUITE 325, HOUSTON, TX ...",[],PROVIDES SECURITY GUARD SERVICES


In [70]:
path = './../Desktop/PDF_extracted'
results = [] 
for file in tqdm(result[result["Directors"].str.contains("\[\]")]['ID Number']):
    pdf = path + '/' + file + '.pdf'
    try:
        with open(pdf, 'rb') as f:
            try:
                rst = read_pdf(f)
            except:
                continue
            try:
                info = handle(rst, 2)
                # use the file name as the ID number
                if info['ID Number'] is None:
                    info['ID Number'] = file
                results.append(info)
            except IndexError:
                # print('list index out of range')
                continue
            except ValueError:
                info = handle(rst, 2)
                # use the file name as the ID number
                if info['ID Number'] is None:
                    info['ID Number'] = file[:-4]
                results.append(info)
                continue
    except:
        continue

100%|████████████████████████████████████████████████████████████████████████████████| 283/283 [01:00<00:00,  4.70it/s]


In [ ]:
new_res = pd.DataFrame(results)
new_res

In [57]:
result = result[result["Directors"].str.contains("\[\]") == False]

In [58]:
new_res = pd.DataFrame(results)
result = pd.concat([result, new_res], ignore_index=True)

In [72]:
pre_res = pd.read_csv('~/Desktop/sec_results4.csv', index_col=0)
pre_res_1 = pd.read_csv('~/Desktop/sec_results4_1.csv', index_col=0)

In [73]:
result = pd.concat([pre_res, pre_res_1], ignore_index=0)
result.to_csv('~/Desktop/sec_results4.csv')

In [62]:
import os
import shutil

record = pd.read_csv('~/Desktop/sec_results9.csv')
for file in tqdm(record['ID Number']):
    pdf = path + '/' + format(file, '09d') + '.pdf'
    if os.path.exists(pdf):
        shutil.move(pdf, './../Desktop/PDF_extracted' + '/' + format(file, '09d') + '.pdf')

100%|████████████████████████████████████████████████████████████████████████████| 3429/3429 [00:01<00:00, 3142.79it/s]
